# Methods: Train and Tune

In [1]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import PolynomialFeatures
from sklearn.tree import DecisionTreeRegressor
from sklearn.feature_selection import SelectKBest, f_regression

## Get back to where we were in EDA

In [2]:
# Set the path to the file you'd like to load
file_path = "student_performance_finalscore.csv"

# Load the latest version
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "sarveshchhetri/student-lifestyle-vs-academic-performance-dataset",
  file_path,
  # Provide any additional arguments like 
  # sql_query or pandas_kwargs. See the 
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

In [3]:
categorical_vars = df.select_dtypes(include=['str']).columns.tolist()
df[categorical_vars] = df[categorical_vars].astype("category")

In [4]:
# sanity check
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 18 columns):
 #   Column                      Non-Null Count  Dtype   
---  ------                      --------------  -----   
 0   Student_ID                  8000 non-null   category
 1   Age                         8000 non-null   int64   
 2   Gender                      8000 non-null   category
 3   Hours_Studied               8000 non-null   float64 
 4   Attendance                  8000 non-null   float64 
 5   Sleep_Hours                 8000 non-null   float64 
 6   Stress_Level                8000 non-null   float64 
 7   Screen_Time                 8000 non-null   float64 
 8   Previous_GPA                8000 non-null   float64 
 9   Part_Time_Job               8000 non-null   category
 10  Study_Method                8000 non-null   category
 11  Diet_Quality                8000 non-null   category
 12  Internet_Quality            8000 non-null   category
 13  Extracurricular             8

## Lets split the data

In [5]:
# now lets split up the dataset into X and y
# also remember that we dont want Age, Student_ID, or Final_Score in X

cols_to_drop = ["Student_ID", "Age", "Gender", "Family_Income_Level"]
# None of these columns are representative of a student's lifestyle choices, they can't control these
target = ["Final_Score"]

X = df.drop(columns=cols_to_drop + target)
y = df[target]


In [6]:
# Split the data into train, tune, and test sets
# 70/15/15 split
X_temp, X_test, y_temp, y_test = train_test_split(X, y, random_state=42, test_size=0.30)
X_train, X_tune, y_train, y_tune = train_test_split(X_temp, y_temp, random_state=42, test_size=0.5)

## Building the Pipeline

In [7]:

# define column groups from X (already has dropped columns)
NUMERIC = X.select_dtypes(include='number').columns.tolist()
CATEGORICAL = X.select_dtypes(include='category').columns.tolist()

In [8]:
# build pipeline
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), NUMERIC),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), CATEGORICAL)
])

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

# fit on train
pipeline.fit(X_train, y_train)

# evaluate on tune set
y_tune_pred = pipeline.predict(X_tune)
print("Tune set R² and RMSE:")
print(r2_score(y_tune, y_tune_pred), "|", np.sqrt(mean_squared_error(y_tune, y_tune_pred)))

Tune set R² and RMSE:
0.6726449868516822 | 7.308716904052848


## Adjusting the Model

### Categorical Model

In [9]:
# Let's tune the model by including only the categorical variables

preprocessor_cat = ColumnTransformer([
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), CATEGORICAL)
])

pipeline_cat = Pipeline([
    ('preprocessor', preprocessor_cat),
    ('model', LinearRegression())
])

pipeline_cat.fit(X_train, y_train)
y_tune_pred_cat = pipeline_cat.predict(X_tune)
print("Tune set R² and RMSE with only categorical variables:")
print(r2_score(y_tune, y_tune_pred_cat), "|", np.sqrt(mean_squared_error(y_tune, y_tune_pred_cat)))

Tune set R² and RMSE with only categorical variables:
0.006394307793473497 | 12.733232749983866


### Numerical Model

In [10]:
# Let's tune the model by including only the numeric variables

preprocessor_num = ColumnTransformer([
    ('num', StandardScaler(), NUMERIC)
])

pipeline_num = Pipeline([
    ('preprocessor', preprocessor_num),
    ('model', LinearRegression())
])

pipeline_num.fit(X_train, y_train)
y_tune_pred_num = pipeline_num.predict(X_tune)
print("Tune set R² and RMSE with only numeric variables:")
print(r2_score(y_tune, y_tune_pred_num), "|", np.sqrt(mean_squared_error(y_tune, y_tune_pred_num)))

Tune set R² and RMSE with only numeric variables:
0.6624036592739152 | 7.4221633365089055


Looks like the numerical features explain the target variable much better than the categorical ones (this was also seen in the EDA stage).

### Polynomial Model

In [11]:
preprocessor_poly = ColumnTransformer([
    ('num', StandardScaler(), NUMERIC),
])

pipeline_poly = Pipeline([
    ('preprocessor', preprocessor_poly),
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('model', LinearRegression())
])

pipeline_poly.fit(X_train, y_train)
y_tune_pred_poly = pipeline_poly.predict(X_tune)
print("Tune set R² and RMSE with polynomial features:")
print(r2_score(y_tune, y_tune_pred_poly), "|", np.sqrt(mean_squared_error(y_tune, y_tune_pred_poly)))

Tune set R² and RMSE with polynomial features:
0.7217595791968883 | 6.73816672838534


In [12]:
# Let's try with all the features and polynomial features

preprocessor_all_poly = ColumnTransformer([
    ('num', StandardScaler(), NUMERIC),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), CATEGORICAL)
])  

pipeline_all_poly = Pipeline([
    ('preprocessor', preprocessor_all_poly),
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('model', LinearRegression())
])

pipeline_all_poly.fit(X_train, y_train)
y_tune_pred_all_poly = pipeline_all_poly.predict(X_tune)
print("Tune set R² and RMSE with all features and polynomial features:")
print(r2_score(y_tune, y_tune_pred_all_poly), "|", np.sqrt(mean_squared_error(y_tune, y_tune_pred_all_poly)))

Tune set R² and RMSE with all features and polynomial features:
0.7262824201995413 | 6.683177377418323


This model seems to perform the best, let's try it on our test set:

In [13]:
preprocessor_best = ColumnTransformer([
    ('num', StandardScaler(), NUMERIC),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), CATEGORICAL)
])  

pipeline_best = Pipeline([
    ('preprocessor', preprocessor_all_poly),
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('model', LinearRegression())
])

pipeline_best.fit(X_train, y_train)
y_test_pred = pipeline_all_poly.predict(X_test)
print("Test set R² and RMSE with all features and polynomial features:")
print(r2_score(y_test, y_test_pred), "|", np.sqrt(mean_squared_error(y_test, y_test_pred)))

Test set R² and RMSE with all features and polynomial features:
0.7318494266732678 | 6.6064956569872
